# Tutorial

In this tutorial, we will walk through the process of creating OME-Zarr datasets using `ome-zarr-converters-tools`.

We cover four approaches:
1. **Table-based HCS plate** (recommended): parse tile metadata from a CSV/DataFrame into an HCS plate
2. **Table-based single images**: same approach for standalone images without plate structure
3. **Manual construction** (advanced): build `Tile` objects programmatically with a custom loader
4. **Fractal integration** (optional): parallel processing with the Fractal platform

All approaches feed into the same conversion pipeline.

For this example we use a [cardiomyocyte differentiation dataset](https://zenodo.org/records/8287221) with:
- 1 well (A/1) in a plate layout
- 3 fields of view (FOVs)
- 2 Z-slices per FOV
- 1 channel (DAPI)

## Step 1: Table-based approach (recommended)

The simplest way to create a converter is to describe your tiles in a CSV table and an `acquisition_details.toml` file.
See the `examples/` directory in the repository for sample table formats.

### CSV table columns

The CSV table must contain one row per tile (one image file on disk). The required columns are:

| Column | Description |
|--------|-------------|
| `file_path` | Path to the raw image file (relative to the `resource` directory or absolute) |
| `fov_id` | Field-of-view identifier (tiles with the same FOV are stitched together) |
| `start_x`, `start_y` | XY position of this tile's top-left corner (in the coordinate system specified by `AcquisitionDetails`) |
| `start_z` | Z position of this tile (slice index or physical position) |
| `length_x`, `length_y` | Tile dimensions in pixels |
| `length_z` | Number of Z slices in this tile (usually 1) |
| `channel_id` | Channel index (0-based) |
| `start_t` | Time-point index |
| `length_t` | Number of time points (usually 1) |

For **HCS plate** data, the table also needs: `plate_row`, `plate_column`, `plate_acquisition`.
For **single images**, it needs: `image_path` (name of the output OME-Zarr dataset).

### 1.1 Load the metadata

First, let's load the CSV table that describes our tiles.

In [ ]:
import pandas as pd

tiles_table = pd.read_csv("../examples/hcs_plate/tiles.csv")
tiles_table

Each row represents one tile (one image file on disk) with its position, size, and metadata.

Next, we define the acquisition details -- pixel sizes, channel info, and coordinate systems.

**Coordinate systems**: The `start_*_coo` parameters tell the library how to interpret position values. Use `"world"` for physical units (micrometers) or `"pixel"` for pixel indices. In this example, `start_x` and `start_y` are in micrometers (world coordinates), while `start_z` and `start_t` are integer indices (pixel coordinates). Lengths are always in pixels.

In [ ]:
from ome_zarr_converters_tools import AcquisitionDetails, ChannelInfo

acq = AcquisitionDetails(
    channels=[ChannelInfo(channel_label="DAPI", wavelength_id="405")],
    pixelsize=0.65,  # micrometers
    z_spacing=5.0,  # micrometers
    t_spacing=1.0,  # seconds
    axes=["t", "c", "z", "y", "x"],
    # Coordinate systems: start positions are in world coordinates,
    # lengths are in pixel coordinates
    start_x_coo="world",
    start_y_coo="world",
    start_z_coo="pixel",
    start_t_coo="pixel",
)
acq

### 1.2 Parse tiles from the table

Use `hcs_images_from_dataframe()` for HCS plate data (or `single_images_from_dataframe()` for standalone images).
This creates a list of `Tile` objects from the table.

In [ ]:
from ome_zarr_converters_tools.core import hcs_images_from_dataframe

tiles = hcs_images_from_dataframe(
    tiles_table=tiles_table,
    acquisition_details=acq,
    plate_name="CardiomyocytePlate",
    acquisition_id=0,
)

fov_names = {t.fov_name for t in tiles}
print(f"Number of tiles: {len(tiles)}")
print(f"FOV names: {fov_names}")
print(f"Collection type: {type(tiles[0].collection).__name__}")
tiles[0]

### 1.3 Aggregate tiles into TiledImages

The `tiles_aggregation_pipeline()` groups tiles that belong to the same image and creates `TiledImage` objects.

The **`resource`** parameter is the base directory for resolving relative file paths. When the CSV contains relative paths like `"image_001.png"`, the `DefaultImageLoader` joins `resource` + `file_path` to find the actual file on disk. If your CSV uses absolute paths, you can omit `resource`.

In [ ]:
from ome_zarr_converters_tools import ConverterOptions, tiles_aggregation_pipeline

data_dir = "../examples/hcs_plate/data"
opts = ConverterOptions()

tiled_images = tiles_aggregation_pipeline(
    tiles=tiles,
    converter_options=opts,
    resource=data_dir,
)

print(f"Number of TiledImages: {len(tiled_images)}")
tiled_image = tiled_images[0]
print(f"Path: {tiled_image.path}")
print(f"Number of regions: {len(tiled_image.regions)}")
print(f"FOV groups: {len(tiled_image.group_by_fov())}")

### 1.4 Build the registration pipeline and write OME-Zarr

The registration pipeline aligns tile positions (e.g., snapping to pixel grid, removing overlaps). Then `tiled_image_creation_pipeline()` writes the final OME-Zarr dataset.

In [ ]:
from ome_zarr_converters_tools.models import (
    AlignmentCorrections,
    OverwriteMode,
    TilingMode,
    WriterMode,
)
from ome_zarr_converters_tools.pipelines import (
    build_default_registration_pipeline,
    tiled_image_creation_pipeline,
)

# Build the default registration pipeline
pipeline = build_default_registration_pipeline(
    alignment_corrections=AlignmentCorrections(),
    tiling_mode=TilingMode.AUTO,
)

# Write the OME-Zarr dataset
zarr_url = "./tutorial_output/output.zarr"
omezarr = tiled_image_creation_pipeline(
    zarr_url=zarr_url,
    tiled_image=tiled_image,
    registration_pipeline=pipeline,
    converter_options=opts,
    writer_mode=WriterMode.BY_FOV,
    overwrite_mode=OverwriteMode.OVERWRITE,
    resource=data_dir,
)

print(f"OME-Zarr written to: {zarr_url}")

### 1.5 Verify the result

The `tiled_image_creation_pipeline()` returns an `OmeZarrContainer` object that can be used to inspect the written dataset with [ngio](https://github.com/fractal-analytics-platform/ngio).

In [ ]:
img = omezarr.get_image()
data = img.get_array()

print(f"Shape (t, c, z, y, x): {data.shape}")
print(f"Channels: {img.channel_labels}")
print(f"Tables: {omezarr.list_tables()}")

## Step 2: Single images (table-based)

The same table-based approach works for standalone images without a plate layout. Use `single_images_from_dataframe()` instead of `hcs_images_from_dataframe()`.

The CSV table needs an `image_path` column instead of `plate_row`/`plate_column`/`plate_acquisition`.

In [ ]:
from ome_zarr_converters_tools.core import single_images_from_dataframe

single_table = pd.read_csv("../examples/single_acquisitions/tiles.csv")
print("Single image CSV columns:", list(single_table.columns))
print()
single_table

In [ ]:
single_tiles = single_images_from_dataframe(
    tiles_table=single_table,
    acquisition_details=acq,
)

print(f"Number of tiles: {len(single_tiles)}")
print(f"Collection type: {type(single_tiles[0].collection).__name__}")
print(f"Image path: {single_tiles[0].collection.image_path}")

## Step 3: Manual tile construction (advanced)

For cases where your data doesn't fit into a CSV table, you can construct `Tile` objects programmatically.
This is useful when:
- Your image format requires a custom loader
- You need fine-grained control over tile construction
- You are integrating with a custom data source

### 3.1 Implement a custom ImageLoader

Any custom loader must extend `ImageLoaderInterface` (a Pydantic model) and implement `load_data()`.
The method should return a NumPy array.

In [ ]:
from typing import Any

import numpy as np
from PIL import Image

from ome_zarr_converters_tools.models._loader import ImageLoaderInterface


class PngLoader(ImageLoaderInterface):
    """Custom loader that loads a single PNG file."""

    file_path: str

    def load_data(self, resource: Any = None) -> np.ndarray:
        """Load the PNG file as a NumPy array."""
        if resource is not None:
            path = f"{resource}/{self.file_path}"
        else:
            path = self.file_path
        return np.array(Image.open(path))

### 3.2 Build Tile objects manually

Each `Tile` needs:
- Position (`start_x`, `start_y`, ...) and size (`length_x`, `length_y`, ...)
- A `collection` defining how the image fits into the output (plate well or standalone)
- An `image_loader` that knows how to load the raw data
- `acquisition_details` with pixel sizes and channel info

In [ ]:
from ome_zarr_converters_tools import SingleImage, Tile

acq_manual = AcquisitionDetails(
    channels=[ChannelInfo(channel_label="DAPI")],
    pixelsize=0.65,
    z_spacing=5.0,
    t_spacing=1.0,
)

collection = SingleImage(image_path="manual_example")

# Build two tiles: FOV_1 with 2 Z-slices
tiles_manual = [
    Tile(
        fov_name="FOV_1",
        start_x=10.0,
        start_y=10.0,
        start_z=0.0,
        length_x=2560,
        length_y=2160,
        length_z=1,
        length_c=1,
        length_t=1,
        collection=collection,
        image_loader=PngLoader(
            file_path="20200812-CardiomyocyteDifferentiation14-Cycle1_B03_T0001F001L01A01Z01C01.png"
        ),
        acquisition_details=acq_manual,
    ),
    Tile(
        fov_name="FOV_1",
        start_x=10.0,
        start_y=10.0,
        start_z=1.0,
        length_x=2560,
        length_y=2160,
        length_z=1,
        length_c=1,
        length_t=1,
        collection=collection,
        image_loader=PngLoader(
            file_path="20200812-CardiomyocyteDifferentiation14-Cycle1_B03_T0001F001L01A01Z02C01.png"
        ),
        acquisition_details=acq_manual,
    ),
]

print(f"Built {len(tiles_manual)} tiles manually")
tiles_manual[0]

### 3.3 Aggregate and write (same pipeline)

From here, the pipeline is the same as the table-based approach.

In [ ]:
# Aggregate
tiled_images_manual = tiles_aggregation_pipeline(
    tiles=tiles_manual,
    converter_options=ConverterOptions(),
    resource=data_dir,
)

# Register and write
zarr_url_manual = "./tutorial_output/manual_output.zarr"
omezarr_manual = tiled_image_creation_pipeline(
    zarr_url=zarr_url_manual,
    tiled_image=tiled_images_manual[0],
    registration_pipeline=pipeline,
    converter_options=ConverterOptions(),
    writer_mode=WriterMode.BY_FOV,
    overwrite_mode=OverwriteMode.OVERWRITE,
    resource=data_dir,
)

# Verify
img_manual = omezarr_manual.get_image()
print(f"Shape: {img_manual.get_array().shape}")
print(f"Channels: {img_manual.channel_labels}")

## Step 4: Fractal integration (optional)

!!! note
    The code below is a **reference sketch** and is not executed by this notebook. It shows the general pattern for integrating with the Fractal platform.

The [Fractal platform](https://fractal-analytics-platform.github.io/fractal-server/) enables parallel processing of conversion tasks.
`ome-zarr-converters-tools` provides utilities to integrate with Fractal's init/compute task model.

The workflow has two phases:

1. **Init task**: sets up the OME-Zarr collection structure (plates/wells) and creates a parallelization list
2. **Compute tasks**: each task converts one `TiledImage` independently (can run in parallel)

```python
from ome_zarr_converters_tools import (
    setup_images_for_conversion,
    generic_compute_task,
    ConvertParallelInitArgs,
    ImageInPlate,
    DefaultImageLoader,
)

# Init task: set up plates and build parallelization list
parallelization_list = setup_images_for_conversion(
    tiled_images=tiled_images,
    zarr_dir="/output/zarr_dir",
    collection_type="ImageInPlate",
    converter_options=opts,
    overwrite_mode=OverwriteMode.OVERWRITE,
)

# Compute tasks: each can run in parallel
for task in parallelization_list:
    generic_compute_task(
        zarr_url=task["zarr_url"],
        init_args=ConvertParallelInitArgs(**task["init_args"]),
        collection_type=ImageInPlate,
        image_loader_type=DefaultImageLoader,
        resource="/path/to/data",
    )
```

The `generic_compute_task()` function handles loading the serialized `TiledImage`, running the registration pipeline, writing the OME-Zarr, and cleaning up temporary files.

## Cleanup

In [ ]:
import shutil

shutil.rmtree("./tutorial_output", ignore_errors=True)
print("Cleaned up tutorial output.")